In [139]:
from statsbombpy import sb
import pandas as pd
import duckdb as db
import plotly.express as px
import warnings
from statsbombpy.api_client import NoAuthWarning
warnings.filterwarnings("ignore", category=NoAuthWarning)
con = db.connect("statsbomb.db")
def query(sql):
    return con.execute(sql).df()






In [141]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

required_keys = ["R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_ACCOUNT_ID", "R2_BUCKET"]
missing = [k for k in required_keys if not os.getenv(k)]

if missing:
    print(f"Missing credentials: {missing}")
else:
    print("All R2 credentials loaded successfully.")

All R2 credentials loaded successfully.


In [143]:
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

account_id = os.getenv("R2_ACCOUNT_ID")
access_key = os.getenv("R2_ACCESS_KEY_ID")
secret_key = os.getenv("R2_SECRET_ACCESS_KEY")
bucket     = os.getenv("R2_BUCKET")

con.execute(f"""
    CREATE OR REPLACE SECRET r2_secret (
        TYPE S3,
        KEY_ID '{access_key}',
        SECRET '{secret_key}',
        ENDPOINT '{account_id}.r2.cloudflarestorage.com',
        REGION 'auto',
        URL_STYLE 'path'
    );
""")

print("httpfs loaded and R2 secret created.")

httpfs loaded and R2 secret created.


In [145]:
con.execute("""
    CREATE OR REPLACE VIEW events_with_comp AS
    SELECT e.*, m.competition_id
    FROM events e
    JOIN matches m ON e.match_id = m.match_id
""")

con.execute(f"""
    COPY events_with_comp
    TO 's3://{bucket}/events'
    (FORMAT PARQUET, PARTITION_BY (competition_id), OVERWRITE_OR_IGNORE TRUE)
""")
print("events written")

con.execute(f"""
    COPY matches
    TO 's3://{bucket}/matches'
    (FORMAT PARQUET, PARTITION_BY (competition_id), OVERWRITE_OR_IGNORE TRUE)
""")
print("matches written")

con.execute(f"""
    COPY competitions
    TO 's3://{bucket}/competitions/competitions.parquet'
    (FORMAT PARQUET)
""")
print("competitions written")

print("All tables written to R2.")

events written


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

matches written
competitions written
All tables written to R2.


In [147]:
con.execute(f"""
    CREATE OR REPLACE VIEW events_r2 AS
    SELECT * FROM read_parquet('s3://{bucket}/events/**/*.parquet',
        hive_partitioning=true)
""")

con.execute(f"""
    CREATE OR REPLACE VIEW matches_r2 AS
    SELECT * FROM read_parquet('s3://{bucket}/matches/**/*.parquet',
        hive_partitioning=true)
""")

con.execute(f"""
    CREATE OR REPLACE VIEW competitions_r2 AS
    SELECT * FROM read_parquet('s3://{bucket}/competitions/competitions.parquet')
""")

print("R2 views created.")

# Smoke test — should return row counts from R2
for view in ["events_r2", "matches_r2", "competitions_r2"]:
    count = con.execute(f"SELECT COUNT(*) FROM {view}").fetchone()[0]
    print(f"  {view}: {count} rows")

R2 views created.
  events_r2: 39101 rows
  matches_r2: 2651 rows
  competitions_r2: 67 rows


In [7]:
comps = sb.competitions()
comps = comps[comps["competition_gender"] == "male"]
comps


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,None,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
70,12,27,Italy,Serie A,male,False,False,2015/2016,2025-08-15T14:28:50.169562,None,None,2025-08-15T14:28:50.169562
71,12,86,Italy,Serie A,male,False,False,1986/1987,2025-11-23T11:00:00.442491,None,None,2025-11-23T11:00:00.442491
73,55,282,Europe,UEFA Euro,male,False,True,2024,2026-05-01T19:54:25.846072,2026-05-01T19:58:06.077979,2026-05-01T19:58:06.077979,2026-05-01T19:54:25.846072
74,55,43,Europe,UEFA Euro,male,False,True,2020,2026-05-10T12:59:26.302088,2026-05-10T13:00:25.564993,2026-05-10T13:00:25.564993,2026-05-10T12:59:26.302088


In [ ]:
all_matches = []

for _, row in comps.iterrows():
    matches = sb.matches(
        competition_id=row["competition_id"],
        season_id=row["season_id"]
    )
    all_matches.append(matches)

matches = pd.concat(all_matches, ignore_index=True)
matches

In [ ]:
all_events = []

for match_id in matches["match_id"][:10]:  
    events = sb.events(match_id=match_id)
    events["match_id"] = match_id
    all_events.append(events)
    print(f"loaded match {match_id} — {len(events)} events")  # progress indicator

events = pd.concat(all_events, ignore_index=True)
print(f"total events: {len(events)}")
events

In [ ]:
events = events.copy()

# location still needs splitting - it comes back as a list
if "location" in events.columns:
    events["location_x"] = events["location"].apply(
        lambda loc: loc[0] if isinstance(loc, list) else None
    )
    events["location_y"] = events["location"].apply(
        lambda loc: loc[1] if isinstance(loc, list) else None
    )
    events = events.drop(columns=["location"])

# statsbombpy already gives these as plain strings - no extraction needed
# just rename for clarity
events = events.rename(columns={
    "type":         "type_name",
    "play_pattern": "play_pattern_name",
    "team":         "team_name",
    "player":       "player_name",
    "position":     "position_name"
})

events = events.copy()
events

In [ ]:
all_lineups = []

for match_id in matches["match_id"][:10]:
    lineup_dict = sb.lineups(match_id=match_id)  # returns {team_name: df}
    for team_name, players in lineup_dict.items():
        players["match_id"] = match_id
        players["team_name"] = team_name
        all_lineups.append(players)

lineups = pd.concat(all_lineups, ignore_index=True)
lineups

In [ ]:
events[["type_name", "player_name", "team_name", "minute", "period", "location_x", "location_y", "match_id"]].head(20)

In [ ]:
for col in matches.columns:
    if col.endswith("_id"):
        matches[col] = matches[col].astype(str)

events.to_parquet("events.parquet", index=False)
matches.to_parquet("matches.parquet", index=False)
comps.to_parquet("comps.parquet", index=False)
lineups.to_parquet("lineups.parquet", index=False)


In [ ]:
con = duckdb.connect("statsbomb.db")
con.execute("CREATE OR REPLACE TABLE events AS SELECT * FROM 'events.parquet'")
con.execute("CREATE OR REPLACE TABLE matches AS SELECT * FROM 'matches.parquet'")
con.execute("CREATE OR REPLACE TABLE competitions AS SELECT * FROM 'comps.parquet'")
con.execute("CREATE OR REPLACE TABLE lineups AS SELECT * FROM 'lineups.parquet'")

In [43]:
def query(sql):
    return con.execute(sql).df()

In [ ]:
query("SELECT * FROM events WHERE type_name = 'Shot' LIMIT 10")


In [ ]:
query("""
SELECT 
player_name as player,
COUNT(player_name) as shots 
FROM events 
WHERE type_name ='Shot'
group by player_name
order by shots desc""")

In [ ]:
query("DESCRIBE events")

In [ ]:
query("""
SELECT 
player_name as player,
COUNT(player_name) as shots 
FROM events 
WHERE type_name ='Shot'
group by player_name
order by shots desc
LIMIT 1""")

In [ ]:
query("DESCRIBE lineups")


In [ ]:
query("""
SELECT 
top_10.name as name,
top_10.shots as shots,
COUNT(cards) as cards
FROM lineups 
RIGHT JOIN (
    SELECT 
    player_id,
    player_name as name,
    COUNT(*) as shots
    FROM events
    WHERE type_name ='Shot'
    GROUP BY player_id, player_name 
    ORDER BY shots desc
    LIMIT 10
)as top_10
ON lineups.player_id = top_10.player_id
GROUP BY top_10.name, top_10.shots
ORDER BY cards desc 



""")

In [ ]:
query("DESCRIBE lineups")


In [ ]:
query("DESCRIBE events")

In [ ]:
'player_id' in events.columns


In [ ]:
query("""

SELECT
l.player_name as name,
COUNT(*) as shots,
COUNT(cards) as cards
FROM lineups l 
JOIN events e ON l.player_id = e.player_id
WHERE e.type_name = 'Shot' 
GROUP by l.player_name
      
      """)

In [ ]:
query("""
SELECT DISTINCT foul_committed_card 
FROM events 
WHERE foul_committed_card IS NOT NULL
""")


In [ ]:
query("""
SELECT DISTINCT bad_behaviour_card
FROM events
WHERE bad_behaviour_card IS NOT NULL
""")

In [39]:
con = db.connect("statsbomb.db")


In [3]:
query("""
SELECT DISTINCT foul_committed_card 
FROM events 
WHERE foul_committed_card IS NOT NULL
""")



,foul_committed_card
0,Red Card
1,Yellow Card
2,Second Yellow


In [7]:
query("""
SELECT * FROM matches
""")

,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,...,home_manager_country_name,away_manager_id,away_manager_name,away_manager_nickname,away_manager_dob,away_manager_country_id,away_manager_country_name,data_version,shot_fidelity_version,xy_fidelity_version
0,3895292,2024-04-06,15:30:00.000,0,1,available,available,2024-07-31T11:03:01.588127,2025-07-06T04:26:07.636270,28,...,Croatia,1000310,Xabier Alonso Olano,Xabi Alonso,1981-11-25,214,Spain,1.1.0,2,2
1,3895320,2024-04-27,19:30:00.000,2,2,available,available,2024-08-06T15:42:54.389576,2025-01-02T11:05:23.462823,31,...,Spain,5213,Sebastian Hoeneß,None,1982-05-12,85,Germany,1.1.0,2,2
2,3895158,2023-12-03,18:30:00.000,1,1,available,available,2024-07-29T12:45:28.152104,2025-09-07T09:35:11.061799,13,...,Spain,5531,Edin Terzić,None,1982-10-30,85,Germany,1.1.0,2,2
3,3895107,2023-10-08,16:30:00.000,3,0,available,available,2024-07-29T12:43:40.606973,2024-10-14T10:35:11.461679,7,...,Spain,104,Steffen Baumgart,None,1972-01-05,85,Germany,1.1.0,2,2
4,3895340,2024-05-12,20:30:00.000,0,5,available,available,2024-07-29T12:16:45.557236,2025-01-01T12:14:22.212709,33,...,Germany,1000310,Xabier Alonso Olano,Xabi Alonso,1981-11-25,214,Spain,1.1.0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2646,3788743,2021-06-12,19:00:00.000,3,0,available,available,2026-05-10T12:57:10.491341,2026-05-10T12:57:51.516855,1,...,Spain,365,Stanislav Cherchesov,None,1963-09-02,188,Russia,1.1.0,2,2
2647,3788744,2021-06-12,13:00:00.000,1,1,available,available,2026-05-09T08:39:44.250015,2026-05-09T08:42:08.909917,1,...,Wales,492,Vladimir Petković,None,1963-08-15,28,Bosnia and Herzegovina,1.1.0,2,2
2648,3887188,1989-05-03,21:30:00.000,2,1,available,unscheduled,2024-02-12T14:45:05.702250,None,12,...,Italy,1003286.0,Arie Haan,None,1948-11-16,85.0,Germany,1.1.0,2,2
2649,3750244,1989-04-05,None,2,0,available,scheduled,2020-07-29T05:00,2021-06-13T16:17:31.694,10,...,None,nan,None,None,None,nan,None,1.1.0,2,2


In [11]:
query("""
SELECT * FROM lineups

""")

,player_id,player_name,player_nickname,jersey_number,country,cards,positions,match_id,team_name
0,3117,Lucas Tousart,None,29,France,"[{'card_type': 'Yellow Card', 'period': 1, 're...","[{'end_reason': 'Tactical Shift', 'from': '00:...",3895292,Union Berlin
1,6985,Robin Gosens,None,6,Germany,"[{'card_type': 'Yellow Card', 'period': 1, 're...","[{'end_reason': 'Tactical Shift', 'from': '00:...",3895292,Union Berlin
2,8239,Robin Knoche,None,31,Germany,[],[],3895292,Union Berlin
3,8397,Kevin Vogt,None,2,Germany,[],"[{'end_reason': 'Tactical Shift', 'from': '00:...",3895292,Union Berlin
4,8933,Alexander Schwolow,None,37,Germany,[],[],3895292,Union Berlin
...,...,...,...,...,...,...,...,...,...
391,42543,Oscar Vilhelmsson,None,29,Sweden,[],[{'end_reason': 'Substitution - Off (Tactical)...,3895220,Darmstadt 98
392,48655,Luca Pfeiffer,None,24,Germany,[],"[{'end_reason': 'Tactical Shift', 'from': '00:...",3895220,Darmstadt 98
393,48825,Andreas Müller,None,16,Germany,[],"[{'end_reason': 'Tactical Shift', 'from': '80:...",3895220,Darmstadt 98
394,153361,Matej Maglica,None,5,Croatia,[],"[{'end_reason': 'Substitution - Off (Injury)',...",3895220,Darmstadt 98


In [ ]:
query("""
    SELECT 
    player_id,
    Player_name,
    jersey_number,
    country,
    team_name
    


""")